# 01 — Exploratory Data Analysis (iterasi 2: EDA terarah)
**Structural Damage Classification — Gempa Gorkha, Nepal 2015**

EDA ini disusun sebagai **daftar pertanyaan**. Setiap jawaban menentukan satu keputusan modeling:

| # | Pertanyaan | Keputusan yang bergantung pada jawabannya |
|---|---|---|
| A | Apakah ada nilai anomali / placeholder (umur 995, tinggi 0, dst.)? | perlu flag/pembersihan atau tidak |
| B | Seberapa banyak bangunan dengan fitur identik tetapi grade berbeda? | batas atas skor (noise label) |
| C | Apakah test diambil acak dari populasi yang sama dengan train? | apakah CV acak bisa dipercaya untuk leaderboard |
| D | Level lokasi mana yang paling informatif dan berapa besar grupnya? | level & smoothing target encoding |
| E | Bagaimana kemiripan grade menurun terhadap jarak nomor ID? Apakah ID 11 digit = rumah tangga? | ukuran tetangga, pembobotan jarak, fitur rumah tangga |
| F | Setelah lokasi dikontrol, fitur konstruksi mana yang masih berpengaruh? | interaksi lokasi × konstruksi yang perlu dibuat |
| G | Apakah proksi fisika (periode alami, kelangsingan) masih berpengaruh setelah lokasi dikontrol? | pertahankan / buang fitur fisika |
| H | Bagaimana distribusi kelas berubah terhadap intensitas wilayah? | strategi macro F1 |

Notebook ini **mandiri** (tidak membutuhkan folder `scripts/`). Gambar disimpan ke `reports/figures/`
dan ringkasan angka ke `reports/eda_findings.json`. **Sel terakhir mencetak ringkasan teks.**

Perkiraan waktu di CPU Kaggle: ±10–15 menit.

In [ ]:
import json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
T0 = time.time()


def find_data_dir():
    """Cari folder berisi train.csv (lokal atau Kaggle)."""
    cands = [Path("../data"), Path("data"),
             Path("/kaggle/input/structural-damage-classification"),
             Path("/kaggle/input/competitions/structural-damage-classification")]
    for c in cands:
        if (c / "train.csv").exists():
            return c
    kin = Path("/kaggle/input")
    if kin.exists():
        for h in sorted(kin.glob("**/train.csv")):
            if (h.parent / "sample_submission.csv").exists():
                return h.parent
    raise FileNotFoundError("train.csv tidak ditemukan")


DATA = find_data_dir()
if Path("../scripts").exists():
    OUT = Path("../reports")
elif Path("/kaggle/working").exists():
    OUT = Path("/kaggle/working/reports")
else:
    OUT = Path("reports")
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
FINDINGS = {}


def save_fig(fig, name):
    """Simpan figure ke reports/figures lalu tampilkan."""
    fig.tight_layout()
    fig.savefig(FIG / name, dpi=120, bbox_inches="tight")
    plt.show()


print("data :", DATA)
print("out  :", OUT)

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")

NUM = ["count_floors_pre_eq", "age_building", "plinth_area_sq_ft", "height_ft_pre_eq"]
LOC = ["district_id", "vdcmun_id", "ward_id"]
CAT = ["land_surface_condition", "foundation_type", "roof_type", "ground_floor_type",
       "other_floor_type", "position", "plan_configuration", "superstructure"]
ID_LEVELS = (2, 4, 6, 8, 9, 10, 11)

y = train["damage_grade"].astype(str).str.extract(r"(\d+)")[0].astype(int).to_numpy()
train["grade"] = y
for df in (train, test):
    sid = df["building_id"].astype(np.int64).astype(str).str.zfill(12)
    df["sid"] = sid
    df["bid"] = df["building_id"].astype(np.int64)
    for k in ID_LEVELS:
        df[f"id{k}"] = sid.str[:k].astype(np.int64)
    df["suffix"] = sid.str[-1].astype(int)
    df["h_per_floor"] = df["height_ft_pre_eq"] / df["count_floors_pre_eq"].clip(lower=1)
    df["period_T"] = 0.075 * (df["height_ft_pre_eq"].clip(lower=1) * 0.3048) ** 0.75
    df["slender"] = df["height_ft_pre_eq"] / np.sqrt(df["plinth_area_sq_ft"].clip(lower=1))

FINDINGS["n_train"], FINDINGS["n_test"] = len(train), len(test)
FINDINGS["class_share"] = {int(k): round(v, 4) for k, v in pd.Series(y).value_counts(normalize=True).sort_index().items()}
print(train.shape, test.shape)
print("distribusi kelas:", FINDINGS["class_share"])
train.head(3)

In [ ]:
# Rata-rata grade bangunan LAIN di area yang sama (leave-one-out) = proksi intensitas guncangan lokal.
for g in ("id6", "id9"):
    grp = train.groupby(g)["grade"]
    s, n = grp.transform("sum"), grp.transform("size")
    train[f"loo_{g}"] = np.where(n > 1, (s - train["grade"]) / (n - 1).clip(lower=1), np.nan)
train["resid"] = train["grade"] - train["loo_id6"]   # sisa grade setelah efek lokasi dikurangi


def effect_table(col, data=None):
    """Rata-rata grade per kategori, sebelum dan sesudah efek lokasi (id6) dikurangi."""
    data = train if data is None else data
    t = data.groupby(col, observed=True).agg(n=("grade", "size"), mean_grade=("grade", "mean"),
                                             resid_after_location=("resid", "mean"))
    return t.sort_values("mean_grade")


def weighted_spread(t, col):
    """Std tertimbang dari rata-rata per kategori (besar = fitur membedakan grade)."""
    t = t.dropna(subset=[col])
    w = t["n"] / t["n"].sum()
    mu = (w * t[col]).sum()
    return float(np.sqrt((w * (t[col] - mu) ** 2).sum()))

## A. Kualitas data & nilai anomali
**Keputusan:** jika kelompok anomali punya grade yang sangat berbeda, tambahkan flag eksplisit;
jika tidak, biarkan (model pohon sudah bisa menangani).

In [ ]:
info = pd.DataFrame({"dtype": train.dtypes.astype(str), "n_unique": train.nunique(), "n_missing": train.isna().sum()})
print("missing total train/test:", int(train.isna().sum().sum()), int(test.isna().sum().sum()))
print("duplikat building_id:", int(train["building_id"].duplicated().sum()),
      "| id test yang ada di train:", int(test["building_id"].isin(train["building_id"]).sum()))
display(info.loc[NUM + LOC + CAT])
display(train[NUM + ["h_per_floor"]].describe(percentiles=[.001, .01, .5, .99, .999]).T.round(2))
print("10 nilai umur paling sering:")
print(train["age_building"].value_counts().head(10).to_string())

In [ ]:
q999 = train["plinth_area_sq_ft"].quantile(0.999)
flags = {
    "age >= 200": train["age_building"] >= 200,
    "age == 0": train["age_building"] == 0,
    "height <= 5 ft": train["height_ft_pre_eq"] <= 5,
    "h_per_floor < 6 ft": train["h_per_floor"] < 6,
    "h_per_floor > 15 ft": train["h_per_floor"] > 15,
    "plinth > p99.9": train["plinth_area_sq_ft"] > q999,
    "floors >= 6": train["count_floors_pre_eq"] >= 6,
}
rows = []
for name, m in flags.items():
    rows.append({"flag": name, "n": int(m.sum()), "share": m.mean(),
                 "mean_grade": train.loc[m, "grade"].mean(), "resid_after_location": train.loc[m, "resid"].mean()})
anom = pd.DataFrame(rows).round(4)
print("rata-rata grade keseluruhan:", round(train["grade"].mean(), 3))
display(anom)
FINDINGS["anomalies"] = anom.set_index("flag")[["n", "mean_grade", "resid_after_location"]].round(3).to_dict("index")

## B. Noise label: fitur identik, grade berbeda?
Bangunan dengan **semua fitur sama** (termasuk ward) tetapi grade berbeda menunjukkan batas atas yang bisa
dicapai model. **Keputusan:** jika persetujuan grade di grup duplikat rendah, gain besar tidak lagi datang
dari fitur bangunan, melainkan dari lokasi yang lebih halus.

In [ ]:
def duplicate_agreement(cols):
    """Untuk baris dengan kombinasi fitur `cols` yang muncul >1 kali: share baris yang sama dengan modus grupnya."""
    cnt = train.groupby(cols + ["grade"], observed=True).size()
    lv = list(range(len(cols)))
    tot = cnt.groupby(level=lv).sum()
    mx = cnt.groupby(level=lv).max()
    dup = tot > 1
    return {"rows_in_dup_groups_share": float(tot[dup].sum() / len(train)),
            "mode_agreement_in_dup_groups": float(mx[dup].sum() / tot[dup].sum())}

dup_res = {
    "fitur_bangunan_saja": duplicate_agreement(NUM + CAT),
    "fitur_bangunan+ward_id": duplicate_agreement(NUM + CAT + ["ward_id"]),
    "fitur_bangunan+id9": duplicate_agreement(NUM + CAT + ["id9"]),
}
dup_df = pd.DataFrame(dup_res).T.round(4)
display(dup_df)
FINDINGS["duplicates"] = dup_df.to_dict("index")

## C. Apakah test berasal dari distribusi yang sama?
1. **Adversarial validation:** model dilatih membedakan baris train vs test. AUC ≈ 0,5 berarti tak bisa
   dibedakan → split acak dan CV acak representatif untuk leaderboard.
2. Fraksi test per wilayah dan korelasinya dengan tingkat kerusakan.
3. Jarak ke tetangga ID terdekat: test→train vs validasi→80% train (cek kepadatan fitur tetangga).

In [ ]:
feat = NUM + LOC + ["bid"] + CAT
both = pd.concat([train[feat].assign(is_test=0), test[feat].assign(is_test=1)], ignore_index=True)
for c in CAT:
    both[c] = both[c].astype("category").cat.codes
Xa, ya = both.drop(columns="is_test"), both["is_test"].to_numpy()
hgb = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1,
                                     categorical_features=[Xa.columns.get_loc(c) for c in CAT], random_state=0)
auc = cross_val_score(hgb, Xa, ya, cv=StratifiedKFold(3, shuffle=True, random_state=0), scoring="roc_auc")
print("adversarial AUC per fold:", auc.round(4), "| mean:", round(auc.mean(), 4))
FINDINGS["adversarial_auc"] = round(float(auc.mean()), 4)

reg = pd.concat([train[["id4"]].assign(t=0), test[["id4"]].assign(t=1)]).groupby("id4")["t"].agg(["mean", "size"])
reg["mean_grade"] = train.groupby("id4")["grade"].mean().reindex(reg.index)
print("\nfraksi test per wilayah (id4):")
print(reg["mean"].describe().round(3).to_string())
corr = reg["mean"].corr(reg["mean_grade"])
print("korelasi fraksi test vs rata-rata grade wilayah:", round(corr, 4))
FINDINGS["test_frac_by_id4"] = {"mean": round(reg["mean"].mean(), 4), "std": round(reg["mean"].std(), 4),
                                "corr_with_grade": round(float(corr), 4)}

In [ ]:
def nearest_same_group(q_b, q_g, k_b, k_g):
    """Jarak |ID| ke bangunan berlabel terdekat dalam grup id6 yang sama (inf jika tidak ada)."""
    order = np.argsort(k_b)
    kb, kg = k_b[order], k_g[order]
    pos = np.searchsorted(kb, q_b)
    best = np.full(len(q_b), np.inf)
    for off in (-1, 0):
        p = np.clip(pos + off, 0, len(kb) - 1)
        d = np.abs(kb[p] - q_b).astype(float)
        d[kg[p] != q_g] = np.inf
        best = np.minimum(best, d)
    return best

rng = np.random.default_rng(0)
val = rng.random(len(train)) < 0.2
tb, tg = train["bid"].to_numpy(), train["id6"].to_numpy()
d_val = nearest_same_group(tb[val], tg[val], tb[~val], tg[~val])
d_test = nearest_same_group(test["bid"].to_numpy(), test["id6"].to_numpy(), tb, tg)
qs = [0.25, 0.5, 0.75, 0.9, 0.99]
nn = pd.DataFrame({"val -> 80% train": np.quantile(d_val[np.isfinite(d_val)], qs),
                   "test -> 100% train": np.quantile(d_test[np.isfinite(d_test)], qs)}, index=qs)
display(nn.round(1))
print("tanpa tetangga di id6 yang sama  val:", round(float(np.mean(~np.isfinite(d_val))), 5),
      " test:", round(float(np.mean(~np.isfinite(d_test))), 5))
FINDINGS["nearest_id_dist_median"] = {"val_80": float(nn.iloc[1, 0]), "test_100": float(nn.iloc[1, 1])}

## D. Granularitas lokasi
Untuk setiap level: jumlah grup, ukuran grup, cakupan di test, dan skor baseline *tebak grade terbanyak
di grup* (out-of-fold). `id11` = 11 digit pertama `building_id` (kandidat "rumah tangga").
**Keputusan:** level yang dipakai di target encoding dan besar smoothing `m` (grup kecil butuh smoothing besar).

In [ ]:
def oof_group_mode(keys, target, n_splits=5, seed=0):
    """Prediksi OOF = grade terbanyak dari grup yang sama di fold lain (fallback: modus global)."""
    keys = np.asarray(keys)
    pred = np.empty_like(target)
    glob = int(np.bincount(target).argmax())
    for tr, va in KFold(n_splits, shuffle=True, random_state=seed).split(keys):
        tab = pd.crosstab(keys[tr], target[tr])
        mode = pd.Series(tab.columns.to_numpy()[tab.to_numpy().argmax(1)], index=tab.index)
        pred[va] = pd.Series(keys[va]).map(mode).fillna(glob).to_numpy()
    return pred

rows = [{"level": "global", "n_groups": 1, "median_size": len(train), "rows_in_groups_lt10": 0.0,
         "test_seen": 1.0, "oof_acc": accuracy_score(y, np.full_like(y, np.bincount(y).argmax())),
         "oof_macro_f1": f1_score(y, np.full_like(y, np.bincount(y).argmax()), average="macro")}]
for lvl in LOC + [f"id{k}" for k in ID_LEVELS]:
    keys = train[lvl]
    sizes = keys.value_counts()
    pred = oof_group_mode(keys.to_numpy(), y)
    rows.append({"level": lvl, "n_groups": len(sizes), "median_size": float(sizes.median()),
                 "rows_in_groups_lt10": float((keys.map(sizes) < 10).mean()),
                 "test_seen": float(test[lvl].isin(sizes.index).mean()),
                 "oof_acc": accuracy_score(y, pred), "oof_macro_f1": f1_score(y, pred, average="macro")})
gran = pd.DataFrame(rows).round(4)
display(gran)
FINDINGS["location_levels"] = gran.set_index("level")[["n_groups", "median_size", "test_seen", "oof_acc",
                                                      "oof_macro_f1"]].to_dict("index")

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(gran["level"][::-1], gran["oof_macro_f1"][::-1], color="#55A868")
ax.set_xlabel("macro F1 out-of-fold (tebak modus grup)")
ax.set_title("Informasi tiap level lokasi")
save_fig(fig, "eda_location_granularity.png")

## E. Struktur nomor ID
1. **Peluruhan kemiripan:** pasangan bangunan dalam id6 yang sama, dikelompokkan menurut selisih nomor ID.
   Dibandingkan dengan pasangan acak dalam id6 yang sama.
2. **Lintas batas:** apakah ID bersebelahan di **id6 berbeda** (tapi id4 sama) masih mirip?
   Apakah kode id6 yang berurutan punya tingkat kerusakan mirip (geografis berurutan)?
3. **Rumah tangga (id11):** apakah bangunan dengan 11 digit pertama sama hampir selalu ber-grade sama?

**Keputusan:** jumlah tetangga `k`, pembobotan jarak, apakah pencarian tetangga boleh melewati batas id6,
dan apakah perlu fitur khusus `id11`.

In [ ]:
def random_pair_agreement(col):
    """Peluang dua bangunan acak (berbeda) di grup `col` yang sama ber-grade sama (rata-rata per baris)."""
    ct = pd.crosstab(train[col], train["grade"])
    n = ct.sum(1)
    same = (ct * (ct - 1)).sum(1)
    tot = n * (n - 1)
    m = n > 1
    return float(((same[m] / tot[m]) * n[m]).sum() / n[m].sum())

srt = train[["bid", "id4", "id6", "grade"]].sort_values("bid")
b, g4, g6, gr = (srt[c].to_numpy() for c in ("bid", "id4", "id6", "grade"))

labels = ["1", "2", "3-5", "6-10", "11-30", "31-100", "101-1000", "1001-10000", ">10000"]
inner_edges = np.array([2, 3, 6, 11, 31, 101, 1001, 10001])
nbin = len(labels)
cnt, agr, adiff = np.zeros(nbin), np.zeros(nbin), np.zeros(nbin)
lag_rows = []
for lag in range(1, 31):
    same = g6[lag:] == g6[:-lag]
    d = (b[lag:] - b[:-lag])[same]
    a = (gr[lag:] == gr[:-lag])[same].astype(float)
    dd = np.abs(gr[lag:] - gr[:-lag])[same].astype(float)
    idx = np.digitize(d, inner_edges)
    cnt += np.bincount(idx, minlength=nbin)
    agr += np.bincount(idx, weights=a, minlength=nbin)
    adiff += np.bincount(idx, weights=dd, minlength=nbin)
    lag_rows.append({"lag": lag, "agree": a.mean(), "mean_abs_diff": dd.mean(), "median_id_dist": float(np.median(d))})

rand6 = random_pair_agreement("id6")
decay = pd.DataFrame({"id_dist": labels, "n_pairs": cnt, "agree": agr / np.maximum(cnt, 1),
                      "mean_abs_grade_diff": adiff / np.maximum(cnt, 1)})
lag_tab = pd.DataFrame(lag_rows)
print("baseline pasangan acak dalam id6:", round(rand6, 4))
display(decay.round(4))
display(lag_tab.iloc[[0, 1, 2, 4, 9, 19, 29]].round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ok = decay["n_pairs"] > 100
axes[0].plot(decay.loc[ok, "id_dist"], decay.loc[ok, "agree"], marker="o")
axes[0].axhline(rand6, ls="--", c="gray", label="pasangan acak id6")
axes[0].set_title("Kemiripan grade vs selisih nomor ID"); axes[0].set_xlabel("selisih ID"); axes[0].legend()
axes[1].plot(lag_tab["lag"], lag_tab["agree"], marker=".")
axes[1].axhline(rand6, ls="--", c="gray")
axes[1].set_title("Kemiripan grade vs urutan tetangga (lag)"); axes[1].set_xlabel("tetangga ke-k")
save_fig(fig, "eda_id_neighbor_decay.png")
FINDINGS["id_decay"] = {"random_pair_id6": round(rand6, 4),
                        "agree_by_dist": dict(zip(labels, (agr / np.maximum(cnt, 1)).round(4).tolist())),
                        "agree_lag1": round(lag_rows[0]["agree"], 4), "agree_lag10": round(lag_rows[9]["agree"], 4),
                        "agree_lag30": round(lag_rows[29]["agree"], 4)}

In [ ]:
# Lintas batas id6
cross = (g6[1:] != g6[:-1]) & (g4[1:] == g4[:-1])
cross_agree = float((gr[1:] == gr[:-1])[cross].mean())
rand4 = random_pair_agreement("id4")
print(f"ID bersebelahan, id6 BERBEDA tapi id4 sama: agree = {cross_agree:.4f} (n={int(cross.sum())}) | acak dalam id4 = {rand4:.4f}")

# Apakah kode id6 berurutan mirip tingkat kerusakannya?
m6 = train.groupby("id6")["grade"].agg(["mean", "size"])
m6 = m6[m6["size"] >= 30].sort_index()
p4 = m6.index.to_numpy() // 100
adj = p4[1:] == p4[:-1]
d_adj = np.abs(np.diff(m6["mean"].to_numpy()))[adj].mean()
shuffled = m6.groupby(p4)["mean"].transform(lambda s: s.sample(frac=1, random_state=0).to_numpy()).to_numpy()
d_rand = np.abs(np.diff(shuffled))[adj].mean()
print(f"|selisih rata-rata grade| id6 berurutan = {d_adj:.4f} vs acak dalam id4 = {d_rand:.4f}")
FINDINGS["cross_boundary"] = {"adjacent_diff_id6_agree": round(cross_agree, 4), "random_id4": round(rand4, 4),
                              "consecutive_id6_mean_absdiff": round(float(d_adj), 4),
                              "shuffled_id6_mean_absdiff": round(float(d_rand), 4)}

In [ ]:
# Rumah tangga: 11 digit pertama
print("distribusi digit terakhir (suffix):")
print(train["suffix"].value_counts().sort_index().to_string())
hh = train.groupby("id11")["grade"].agg(["size", "nunique"])
multi = hh[hh["size"] > 1]
rand11 = random_pair_agreement("id11")
res_hh = {
    "groups_with_2plus": int(len(multi)),
    "rows_in_2plus_share": round(float(multi["size"].sum() / len(train)), 4),
    "groups_all_same_grade": round(float((multi["nunique"] == 1).mean()), 4),
    "pair_agreement_id11": round(rand11, 4),
    "pair_agreement_id6": round(rand6, 4),
    "test_rows_sharing_id11_with_train": round(float(test["id11"].isin(set(train["id11"])).mean()), 4),
}
print(json.dumps(res_hh, indent=1))
FINDINGS["household_id11"] = res_hh

# Apakah grade berbeda antar suffix (bangunan utama vs tambahan)?
display(train.groupby("suffix").agg(n=("grade", "size"), mean_grade=("grade", "mean"),
                                    resid_after_location=("resid", "mean")).round(3))

## F. Efek konstruksi setelah lokasi dikontrol
`resid_after_location` = grade − rata-rata grade bangunan lain di id6 yang sama.
Nilai negatif = lebih tahan dari tetangganya; positif = lebih rentan.
`spread_before` vs `spread_after` mengukur seberapa kuat fitur membedakan grade sebelum/sesudah lokasi dikontrol.

In [ ]:
train["floors_bin"] = train["count_floors_pre_eq"].clip(upper=6)
train["age_bin"] = pd.cut(train["age_building"], [-1, 5, 10, 15, 20, 30, 50, 100, 10_000])
train["area_bin"] = pd.qcut(train["plinth_area_sq_ft"], 8, duplicates="drop")
rank_rows = []
for col in CAT + ["floors_bin", "age_bin", "area_bin"]:
    t = effect_table(col)
    rank_rows.append({"feature": col, "n_levels": len(t), "spread_before": weighted_spread(t, "mean_grade"),
                      "spread_after_location": weighted_spread(t, "resid_after_location")})
rank = pd.DataFrame(rank_rows).sort_values("spread_after_location", ascending=False).round(4)
display(rank)
FINDINGS["construction_effect_rank"] = rank.set_index("feature")[["spread_before", "spread_after_location"]].to_dict("index")

In [ ]:
for col in ["superstructure", "foundation_type", "roof_type", "ground_floor_type", "other_floor_type"]:
    print(f"\n=== {col} ===")
    display(effect_table(col).round(3))

### F.2 Kurva kerentanan (fragility) empiris
Wilayah dibagi menjadi 8 kelompok menurut intensitas lokal (rata-rata grade tetangga id6, leave-one-out).
Untuk tiap material: rata-rata grade di setiap kelompok intensitas. Garis yang **tidak sejajar** berarti ada
interaksi lokasi × material yang harus dimodelkan.

In [ ]:
train["intensity_bin"] = pd.qcut(train["loo_id6"], 8, labels=False, duplicates="drop")
top_mat = train["superstructure"].value_counts()
top_mat = top_mat[top_mat >= 3000].index
frag = train[train["superstructure"].isin(top_mat)].groupby(["intensity_bin", "superstructure"])["grade"].mean().unstack()
fig, ax = plt.subplots(figsize=(9, 5))
for c in frag.columns:
    ax.plot(frag.index, frag[c], marker="o", label=c.replace("has_superstructure_", ""))
ax.set_xlabel("kelompok intensitas wilayah (0 = paling ringan)"); ax.set_ylabel("rata-rata grade")
ax.set_title("Kurva kerentanan empiris per material"); ax.legend(fontsize=8)
save_fig(fig, "eda_fragility_curves.png")
gap = (frag.max(axis=1) - frag.min(axis=1)).round(3)
print("selisih material terkuat-terlemah per kelompok intensitas:")
print(gap.to_string())
FINDINGS["fragility_gap_by_intensity"] = {int(k): float(v) for k, v in gap.items()}

## G. Proksi fisika setelah lokasi dikontrol

In [ ]:
phys_rows = []
for col in ["period_T", "slender", "h_per_floor"]:
    b_ = pd.qcut(train[col], 10, duplicates="drop")
    t = train.groupby(b_, observed=True).agg(n=("grade", "size"), mean_grade=("grade", "mean"),
                                             resid_after_location=("resid", "mean"))
    phys_rows.append({"feature": col, "spread_before": weighted_spread(t, "mean_grade"),
                      "spread_after_location": weighted_spread(t, "resid_after_location")})
    print(f"\n=== {col} (desil) ===")
    display(t.round(3))
phys = pd.DataFrame(phys_rows).round(4)
display(phys)
FINDINGS["physics_proxies"] = phys.set_index("feature").to_dict("index")

## H. Distribusi kelas vs intensitas wilayah
Macro F1 menuntut setiap kelas terprediksi dengan baik. Tabel ini menunjukkan di intensitas mana tiap grade
muncul: jika Grade 2–3 tersebar di semua intensitas, mereka sulit dipisahkan hanya dari lokasi.

In [ ]:
cls = pd.crosstab(train["intensity_bin"], train["grade"], normalize="index").round(3)
display(cls)
fig, ax = plt.subplots(figsize=(9, 4))
left = np.zeros(len(cls))
for g in cls.columns:
    ax.barh(cls.index.astype(str), cls[g], left=left, label=f"Grade {g}")
    left += cls[g].to_numpy()
ax.set_xlabel("proporsi"); ax.set_ylabel("kelompok intensitas"); ax.legend(ncol=5, fontsize=8, loc="lower center",
                                                                             bbox_to_anchor=(0.5, -0.35))
ax.set_title("Komposisi grade per intensitas wilayah")
save_fig(fig, "eda_class_by_intensity.png")
sd9 = train.groupby("id9")["grade"].std().dropna()
print("std grade dalam blok id9 (median):", round(float(sd9.median()), 3))
FINDINGS["class_by_intensity"] = {int(k): v for k, v in cls.to_dict("index").items()}
FINDINGS["median_std_grade_in_id9"] = round(float(sd9.median()), 3)

## Ringkasan
Sel di bawah menyimpan `reports/eda_findings.json` dan mencetak ringkasan teks.
**Salin seluruh output sel terakhir** (atau kirim file JSON-nya) untuk menentukan perubahan di iterasi berikutnya.

In [ ]:
def _clean(o):
    """Ubah tipe numpy/pandas ke tipe JSON biasa."""
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else round(float(o), 4)
    return o

FINDINGS = _clean(FINDINGS)
with open(OUT / "eda_findings.json", "w") as fh:
    json.dump(FINDINGS, fh, indent=1)
print(f"selesai dalam {(time.time() - T0) / 60:.1f} menit -> {OUT / 'eda_findings.json'}\n")
print("===== EDA SUMMARY (salin semua di bawah ini) =====")
print(json.dumps(FINDINGS, separators=(",", ":")))